# Run the below cell and the GUI will open

In [ ]:
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk
import torch
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')

# Load the trained model
model = models.resnet18(pretrained=True)
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, 2)
)
model.load_state_dict(torch.load(r"D:\Education\كلية الهندسة جامعة حلوان\فرقة ثالثة - هندسة طبية\الترم الأول\Pattern Recognition\Project\Chest X-Ray Images (Pneumonia) Classification DL Model\pneumonia_model_by_team10.pth", map_location=torch.device('cpu')))
model.eval()

# Transforms for the input image
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Function to predict the image class
def predict_image(image_path):
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0)  # إضافة البعد الأول
    with torch.no_grad():
        outputs = model(image_tensor)
        _, predicted = torch.max(outputs, 1)
    return "Pneumonia detected :(" if predicted.item() == 1 else "Normal :)"

# Create the GUI
def create_gui():
    # Main window
    root = tk.Tk()
    root.title("Pneumonia Detection Model ==BY TEAM 10==")

    # Function to load and display the image
    def show_image():
        global img_path, img_display
        img_path = filedialog.askopenfilename(filetypes=[("Image files", "*.jpg;*.jpeg;*.png")])
        if img_path:
            img = Image.open(img_path)
            img = img.resize((300, 300))  # تغيير حجم الصورة للعرض
            img_display = ImageTk.PhotoImage(img)
            panel.configure(image=img_display)
            panel.image = img_display

    # Function to analyze the image by the model
    def analyze_image():
        if img_path:
            result = predict_image(img_path)
            result_label.config(text=f"Result: {result}")

    # GUI frame
    panel = tk.Label(root)
    panel.pack(pady=50)

    # Upload image button
    load_button = tk.Button(root, text="Load X-Ray Image", command=show_image, height=2, width=15)
    load_button.pack(pady=5)

    # Analyze image button
    analyze_button = tk.Button(root, text="Analyze The Image", command=analyze_image, height=2, width=15)
    analyze_button.pack(pady=5)

    # Show result label
    result_label = tk.Label(root, text="Result: ", font=("Arial", 16))
    result_label.pack(pady=20)

    # Run the GUI
    root.mainloop()

# Run the GUI
create_gui()
